# GamePulse — 03 · Gold Layer
## Joins, Aggregations & Analysis-Ready Tables

**SYST52461 – Big Data Storage and Analysis**  
**Term Project · Winter 2026**

---

### Purpose
This notebook reads the four Silver tables, joins and aggregates them into three **Gold** Delta tables  
that are optimized for dashboard visualizations and EDA.

### Gold Tables Produced
| Table | Description |
|---|---|
| `player_summary_gold` | One row per player — lifetime sessions, total playtime, total spend, avg score |
| `game_engagement_gold` | One row per game — total sessions, avg session duration, avg score, win rate |
| `revenue_trend_gold` | Monthly aggregation — total revenue, total sessions, and unique active players |

## 1. Environment Setup

In [0]:
import logging
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("gamepulse.gold")

CATALOG = "sandbox_catalog"
SCHEMA  = "gamepulse"

log.info("Gold notebook ready.")

14:20:22  INFO  Gold notebook ready.


## 2. Helper Utilities

In [0]:
def read_table(table_name: str):
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df = spark.read.table(full_name)
    log.info("Loaded  <- %s  (%d rows)", full_name, df.count())
    return df


def write_table(df, table_name: str) -> None:
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    log.info("Written -> %s  (%d rows)", full_name, df.count())


log.info("Helpers defined.")

14:20:22  INFO  Received command c on object id p0
14:20:23  INFO  Helpers defined.


## 3. Load Silver Tables

In [0]:
players_s   = read_table("players_silver")
games_s     = read_table("games_silver")
sessions_s  = read_table("game_sessions_silver")
purchases_s = read_table("purchases_silver")

14:20:23  INFO  Received command c on object id p0
14:20:24  INFO  Loaded  <- sandbox_catalog.gamepulse.players_silver  (468 rows)
14:20:24  INFO  Loaded  <- sandbox_catalog.gamepulse.games_silver  (27 rows)
14:20:25  INFO  Loaded  <- sandbox_catalog.gamepulse.game_sessions_silver  (2944 rows)
14:20:25  INFO  Loaded  <- sandbox_catalog.gamepulse.purchases_silver  (1919 rows)


---
## 4. Build `player_summary_gold`

**Logic:**
1. Aggregate `game_sessions_silver` by `player_id` → lifetime session stats
2. Aggregate `purchases_silver` by `player_id` → lifetime spend stats
3. Join both aggregations to `players_silver` demographics

**Key columns produced:**
- `total_sessions`, `total_playtime_min`, `avg_score`, `win_rate`
- `total_purchases`, `total_spent`, `avg_purchase_value`
- `region`, `account_type`, `age`

In [0]:
# ── Step 1: Session aggregation per player ─────────────────────────────────────
session_agg = (
    sessions_s
    .groupBy("player_id")
    .agg(
        F.count("session_id").alias("total_sessions"),
        F.round(F.sum("duration_minutes"), 1).alias("total_playtime_min"),
        F.round(F.avg("score"), 1).alias("avg_score"),
        F.round(
            F.count(F.when(F.col("outcome") == "Win", True)) /
            F.count("session_id") * 100,
            1
        ).alias("win_rate_pct"),
        F.max("session_date").alias("last_session_date"),
    )
)

log.info("session_agg: %d player rows", session_agg.count())

14:20:27  INFO  session_agg: 499 player rows


In [0]:
# ── Step 2: Purchase aggregation per player ────────────────────────────────────
purchase_agg = (
    purchases_s
    .groupBy("player_id")
    .agg(
        F.count("purchase_id").alias("total_purchases"),
        F.round(F.sum("amount_spent"), 2).alias("total_spent"),
        F.round(F.avg("amount_spent"), 2).alias("avg_purchase_value"),
        F.max("purchase_date").alias("last_purchase_date"),
    )
)

log.info("purchase_agg: %d player rows", purchase_agg.count())

14:20:28  INFO  Received command c on object id p0
14:20:28  INFO  purchase_agg: 486 player rows


In [0]:
# ── Step 3: Join everything to player demographics ─────────────────────────────
player_summary_gold = (
    players_s
    .select("player_id", "username", "age", "region", "account_type", "join_date")
    .join(session_agg,  on="player_id", how="left")
    .join(purchase_agg, on="player_id", how="left")
    # Players with no sessions or purchases get 0 instead of null
    .fillna({
        "total_sessions":    0,
        "total_playtime_min": 0.0,
        "avg_score":          0.0,
        "win_rate_pct":       0.0,
        "total_purchases":   0,
        "total_spent":        0.0,
        "avg_purchase_value": 0.0,
    })
)

print(f"player_summary_gold: {player_summary_gold.count()} rows")
display(player_summary_gold.orderBy("total_spent", ascending=False).limit(15))

player_summary_gold: 468 rows


player_id,username,age,region,account_type,join_date,total_sessions,total_playtime_min,avg_score,win_rate_pct,last_session_date,total_purchases,total_spent,avg_purchase_value,last_purchase_date
357,user_0357,26,Latin America,Premium,2025-10-02,1,160.8,7242.0,100.0,2025-12-06,11,591.46,53.77,2025-11-30
379,user_0379,23,Asia Pacific,Pro,2025-09-18,6,303.0,3677.5,16.7,2025-12-09,9,525.07,58.34,2025-10-05
350,user_0350,23,North America,Free,2025-06-16,8,977.2,4641.6,25.0,2025-08-22,9,493.22,54.8,2025-08-15
228,user_0228,44,Latin America,Pro,2024-07-26,6,577.2,5208.7,16.7,2025-12-28,10,492.72,49.27,2025-10-06
232,user_0232,26,Latin America,Free,2023-04-14,5,311.1,2188.2,40.0,2024-05-07,8,492.3,61.54,2025-11-17
199,user_0199,32,North America,Free,2024-10-26,6,634.0,5446.5,33.3,2025-12-20,7,486.24,69.46,2025-11-29
285,user_0285,49,Latin America,Pro,2024-10-23,4,233.4,6845.5,50.0,2025-04-15,8,478.9,59.86,2025-12-24
318,user_0318,18,Europe,Pro,2024-12-16,4,311.9,5643.5,50.0,2025-06-26,8,462.46,57.81,2025-12-06
111,user_0111,40,Asia Pacific,Free,2025-03-07,7,883.6,6079.9,0.0,2025-09-23,7,454.65,64.95,2025-09-17
225,user_0225,15,Europe,Pro,2023-08-26,4,364.4,3091.0,25.0,2025-12-09,7,439.32,62.76,2025-07-11


---
## 5. Build `game_engagement_gold`

**Logic:**
1. Aggregate `game_sessions_silver` by `game_id` → engagement metrics
2. Join to `games_silver` for game title, genre, and mode

In [0]:
# Aggregate session metrics by game
game_agg = (
    sessions_s
    .groupBy("game_id")
    .agg(
        F.count("session_id").alias("total_sessions"),
        F.countDistinct("player_id").alias("unique_players"),
        F.round(F.avg("duration_minutes"), 1).alias("avg_duration_min"),
        F.round(F.avg("score"), 1).alias("avg_score"),
        F.round(F.sum("duration_minutes"), 1).alias("total_playtime_min"),
        F.round(
            F.count(F.when(F.col("outcome") == "Win", True)) /
            F.count("session_id") * 100,
            1
        ).alias("win_rate_pct"),
    )
)

# Join to game metadata
game_engagement_gold = (
    games_s
    .select("game_id", "title", "genre", "game_mode", "release_year")
    .join(game_agg, on="game_id", how="left")
    .fillna({"total_sessions": 0, "unique_players": 0})
    .orderBy("total_sessions", ascending=False)
)

print(f"game_engagement_gold: {game_engagement_gold.count()} rows")
display(game_engagement_gold)

game_engagement_gold: 27 rows


game_id,title,genre,game_mode,release_year,total_sessions,unique_players,avg_duration_min,avg_score,total_playtime_min,win_rate_pct
6,Frost Arena,Strategy,Multiplayer,2023,117,101,98.6,5097.4,11533.0,25.6
23,Titan Falls,Strategy,Co-op,2020,115,103,84.1,5455.8,9676.8,23.5
1,Shadow Realm,Strategy,Solo,2022,111,104,91.2,5051.6,10127.0,30.6
13,Terra Wars,Puzzle,Co-op,2022,110,98,93.8,4957.3,10323.2,26.4
5,Star Siege,Puzzle,Co-op,2019,106,97,83.4,4771.0,8839.6,23.6
27,Night Patrol,Horror,Solo,2023,105,100,90.7,5044.2,9526.2,30.5
18,Storm Legends,Sports,Solo,2025,105,95,87.3,4891.5,9171.6,25.7
4,Dungeon Lords,Horror,Solo,2025,103,93,90.2,4694.7,9292.9,18.4
7,Turbo Clash,Strategy,Solo,2019,101,97,90.1,5074.1,9096.2,23.8
20,Solar Clash,Rpg,Solo,2020,99,90,85.2,4653.6,8434.9,19.2


---
## 6. Build `revenue_trend_gold`

**Logic:**  
Aggregate `purchases_silver` and `game_sessions_silver` by calendar month  
to produce a monthly time series of revenue and engagement activity.

In [0]:
# Monthly revenue from purchases
monthly_revenue = (
    purchases_s
    .withColumn("month",  F.date_format("purchase_date", "yyyy-MM"))
    .groupBy("month")
    .agg(
        F.round(F.sum("amount_spent"), 2).alias("total_revenue"),
        F.count("purchase_id").alias("total_purchases"),
        F.countDistinct("player_id").alias("paying_players"),
    )
)

# Monthly session activity
monthly_sessions = (
    sessions_s
    .withColumn("month", F.date_format("session_date", "yyyy-MM"))
    .groupBy("month")
    .agg(
        F.count("session_id").alias("total_sessions"),
        F.countDistinct("player_id").alias("active_players"),
        F.round(F.avg("duration_minutes"), 1).alias("avg_session_duration_min"),
    )
)

# Join on month
revenue_trend_gold = (
    monthly_revenue
    .join(monthly_sessions, on="month", how="outer")
    .orderBy("month")
)

print(f"revenue_trend_gold: {revenue_trend_gold.count()} rows (months)")
display(revenue_trend_gold)

revenue_trend_gold: 36 rows (months)


month,total_revenue,total_purchases,paying_players,total_sessions,active_players,avg_session_duration_min
2023-01,2508.14,54,50,75,69,88.0
2023-02,2785.2,55,55,72,66,90.1
2023-03,3120.5,55,52,92,83,100.6
2023-04,2582.15,52,47,91,81,84.5
2023-05,2453.94,47,46,94,85,95.6
2023-06,2656.82,50,45,75,70,88.6
2023-07,2145.42,46,44,92,80,90.9
2023-08,2676.85,53,52,85,76,82.9
2023-09,2025.26,43,40,83,79,89.9
2023-10,2488.23,51,49,85,77,95.4


---
## 7. Write Gold Tables to Delta

In [0]:
write_table(player_summary_gold,  "player_summary_gold")
write_table(game_engagement_gold, "game_engagement_gold")
write_table(revenue_trend_gold,   "revenue_trend_gold")

log.info("All Gold tables written successfully.")

14:20:48  INFO  Received command c on object id p0
14:20:52  INFO  Written -> sandbox_catalog.gamepulse.player_summary_gold  (468 rows)
14:20:58  INFO  Written -> sandbox_catalog.gamepulse.game_engagement_gold  (27 rows)
14:21:03  INFO  Written -> sandbox_catalog.gamepulse.revenue_trend_gold  (36 rows)
14:21:03  INFO  All Gold tables written successfully.


---
## 8. Gold Layer Summary

In [0]:
print("=" * 50)
print("Gold Layer Summary")
print("=" * 50)
for tbl in ["player_summary_gold", "game_engagement_gold", "revenue_trend_gold"]:
    df = spark.read.table(f"{CATALOG}.{SCHEMA}.{tbl}")
    print(f"  {tbl:<30} {df.count():>5} rows")
print("=" * 50)

Gold Layer Summary
  player_summary_gold              468 rows
  game_engagement_gold              27 rows
  revenue_trend_gold                36 rows
